# ASCENT-G H2 Retention Eval — MCQ Batch

**Experiment:** H2 — Do Phase 1 LoRA adapters (trained on Instruct) transfer to the Base model?

**Tasks:** ARC-Easy, ARC-Challenge, OpenbookQA, CommonsenseQA

**Protocol:**
1. Load adapter on `Qwen2.5-1.5B-Instruct` → greedy eval on 200 validation examples → `instruct_acc`
2. Load same adapter on `Qwen2.5-1.5B` (base) with Instruct tokenizer → same 200 examples → `base_acc`
3. `retention = base_acc / instruct_acc`

**Decision criteria:** Pass if retention > 0.70, Fail if < 0.30

**Adapters:** `/kaggle/input/ascent-g-phase1-adapters/{task_name}/adapter/`

**GPU:** T4 (fp16, autocast_adapter_dtype=False)

In [ ]:
import subprocess, sys, torch

torch_ver = torch.__version__
torch_base = torch_ver.split("+")[0]

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchao>=0.16.0", "--no-deps"], check=True)

constraints = (
    f"torch=={torch_base}\n"
    "torchvision\n"
    "torchaudio\n"
    "torchao\n"
).encode()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "trl==0.17.0", "transformers==4.51.3",
                "peft==0.14.0", "accelerate", "datasets",
                "--constraint", "/dev/stdin"],
               input=constraints, check=True)

import importlib
for mod in ["trl", "transformers", "peft", "datasets"]:
    importlib.invalidate_caches()

import trl, transformers, peft, datasets
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — abort"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU model  : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"bf16       : {'supported' if bf16_supported else 'NOT supported — will use float16'}")
print(f"torch      : {torch.__version__}")

REQUIRED_GPU = "T4"
if REQUIRED_GPU not in gpu_name:
    raise RuntimeError(
        f"Wrong GPU: got '{gpu_name}', need {REQUIRED_GPU}. "
        f"Go to Session options → Accelerator → GPU T4 x2 and re-run."
    )

MODEL_DTYPE = torch.bfloat16 if bf16_supported else torch.float16

HW_META = {
    "gpu_model": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "bf16_supported": bf16_supported,
    "torch_version": torch.__version__,
}
print("\nHW_META:", HW_META)
print(f"MODEL_DTYPE: {MODEL_DTYPE}")

In [ ]:
import re, gc, json, datetime
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

# ── Constants ──────────────────────────────────────────────────────────────
INSTRUCT_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_MODEL_ID     = "Qwen/Qwen2.5-1.5B"
TOKENIZER_ID      = "Qwen/Qwen2.5-1.5B-Instruct"  # used for BOTH models

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

ADAPTER_BASE = "/kaggle/input/ascent-g-phase1-adapters"
N_EVAL = 200
MAX_NEW_TOKENS_MCQ = 64

SYSTEM_PROMPT = (
    "You are a reasoning assistant. "
    "Choose the best answer option and make the final choice explicit as The answer is <option>."
)

# ── Shared tokenizer (loaded once) ─────────────────────────────────────────
print(f"Loading tokenizer from {TOKENIZER_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
print("Tokenizer loaded.")

# ── Helpers ────────────────────────────────────────────────────────────────
def extract_final_option(text):
    """Extract the last answer option label (A-E) from a completion."""
    matches = re.findall(r"\b([A-E])\b", text.upper())
    return matches[-1] if matches else None


def load_model_with_adapter(model_id, adapter_path):
    """Load a causal LM and attach a LoRA adapter. Returns PeftModel."""
    print(f"  Loading base: {model_id}")
    base = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=MODEL_DTYPE,
        device_map="auto",
    )
    print(f"  Attaching adapter: {adapter_path}")
    model = PeftModel.from_pretrained(base, adapter_path, autocast_adapter_dtype=False)
    model.eval()
    return model


def unload_model(model):
    """Delete model and free GPU memory."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("  Model unloaded, cache cleared.")


def greedy_eval_mcq(model, prompts, answers):
    """
    Greedy MCQ evaluation.
    prompts  : list of already-formatted chat strings
    answers  : list of ground-truth option labels (A-E)
    Returns accuracy (float).
    """
    correct = 0
    for prompt, gold in zip(prompts, answers):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS_MCQ,
                do_sample=False,
            )
        completion = tokenizer.decode(
            output[0][input_len:], skip_special_tokens=True
        )
        pred = extract_final_option(completion)
        gold_opt = extract_final_option(str(gold))
        if pred is not None and gold_opt is not None and pred == gold_opt:
            correct += 1
    return correct / len(prompts) if prompts else 0.0


def compute_retention(base_acc, instruct_acc):
    """Retention = base_acc / instruct_acc. Returns (retention, verdict)."""
    if instruct_acc == 0.0:
        return None, "UNDEFINED (instruct_acc=0)"
    retention = base_acc / instruct_acc
    if retention >= 0.70:
        verdict = "PASS"
    elif retention < 0.30:
        verdict = "FAIL"
    else:
        verdict = "MARGINAL"
    return retention, verdict


print("Helper functions defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 1: ARC-Easy
# Phase 1 instruct best_reward: 1.0000
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "arc-easy"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"
PHASE1_BEST_REWARD = 1.0000

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Phase 1 best_reward (instruct): {PHASE1_BEST_REWARD}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: allenai/ai2_arc, ARC-Easy, validation ...")
ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="validation")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_arc(example):
    question = str(example["question"])
    choices = example["choices"]
    labels = choices["label"]
    texts = choices["text"]
    rendered = "\n".join(f"{label}. {text}" for label, text in zip(labels, texts))
    user_prompt = f"{question}\n\nOptions:\n{rendered}"
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt, "answer": str(example["answerKey"]).upper()}

ds_fmt = ds.map(format_arc)
prompts = ds_fmt["prompt"]
answers = ds_fmt["answer"]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
arc_easy_instruct_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  instruct_acc = {arc_easy_instruct_acc:.4f}")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
arc_easy_base_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  base_acc = {arc_easy_base_acc:.4f}")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
arc_easy_retention, arc_easy_verdict = compute_retention(arc_easy_base_acc, arc_easy_instruct_acc)
print(f"\n{'─'*40}")
print(f"ARC-Easy  instruct_acc : {arc_easy_instruct_acc:.4f}")
print(f"ARC-Easy  base_acc     : {arc_easy_base_acc:.4f}")
print(f"ARC-Easy  retention    : {arc_easy_retention:.4f}")
print(f"ARC-Easy  verdict      : {arc_easy_verdict}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 2: ARC-Challenge
# Phase 1 instruct best_reward: unknown — fresh eval used
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "arc-challenge"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: allenai/ai2_arc, ARC-Challenge, validation ...")
ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="validation")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

ds_fmt = ds.map(format_arc)
prompts = ds_fmt["prompt"]
answers = ds_fmt["answer"]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
arc_challenge_instruct_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  instruct_acc = {arc_challenge_instruct_acc:.4f}")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
arc_challenge_base_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  base_acc = {arc_challenge_base_acc:.4f}")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
arc_challenge_retention, arc_challenge_verdict = compute_retention(arc_challenge_base_acc, arc_challenge_instruct_acc)
print(f"\n{'─'*40}")
print(f"ARC-Challenge  instruct_acc : {arc_challenge_instruct_acc:.4f}")
print(f"ARC-Challenge  base_acc     : {arc_challenge_base_acc:.4f}")
print(f"ARC-Challenge  retention    : {arc_challenge_retention:.4f}")
print(f"ARC-Challenge  verdict      : {arc_challenge_verdict}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 3: OpenbookQA
# Phase 1 instruct best_reward: 0.9250
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "openbookqa"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"
PHASE1_BEST_REWARD = 0.9250

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Phase 1 best_reward (instruct): {PHASE1_BEST_REWARD}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: allenai/openbookqa, main, validation ...")
ds = load_dataset("allenai/openbookqa", "main", split="validation")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_openbookqa(example):
    question = str(example["question_stem"])
    choices = example["choices"]
    labels = choices["label"]
    texts = choices["text"]
    rendered = "\n".join(f"{label}. {text}" for label, text in zip(labels, texts))
    user_prompt = f"{question}\n\nOptions:\n{rendered}"
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt, "answer": str(example["answerKey"]).upper()}

ds_fmt = ds.map(format_openbookqa)
prompts = ds_fmt["prompt"]
answers = ds_fmt["answer"]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
openbookqa_instruct_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  instruct_acc = {openbookqa_instruct_acc:.4f}")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
openbookqa_base_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  base_acc = {openbookqa_base_acc:.4f}")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
openbookqa_retention, openbookqa_verdict = compute_retention(openbookqa_base_acc, openbookqa_instruct_acc)
print(f"\n{'─'*40}")
print(f"OpenbookQA  instruct_acc : {openbookqa_instruct_acc:.4f}")
print(f"OpenbookQA  base_acc     : {openbookqa_base_acc:.4f}")
print(f"OpenbookQA  retention    : {openbookqa_retention:.4f}")
print(f"OpenbookQA  verdict      : {openbookqa_verdict}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 4: CommonsenseQA
# Phase 1 instruct best_reward: unknown — fresh eval used
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "commonsenseqa"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: tau/commonsense_qa, validation ...")
ds = load_dataset("tau/commonsense_qa", split="validation")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_commonsenseqa(example):
    question = str(example["question"])
    choices = example["choices"]
    labels = choices["label"]
    texts = choices["text"]
    rendered = "\n".join(f"{label}. {text}" for label, text in zip(labels, texts))
    user_prompt = f"{question}\n\nOptions:\n{rendered}"
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt, "answer": str(example["answerKey"]).upper()}

ds_fmt = ds.map(format_commonsenseqa)
prompts = ds_fmt["prompt"]
answers = ds_fmt["answer"]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
commonsenseqa_instruct_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  instruct_acc = {commonsenseqa_instruct_acc:.4f}")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
commonsenseqa_base_acc = greedy_eval_mcq(model, prompts, answers)
print(f"  base_acc = {commonsenseqa_base_acc:.4f}")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
commonsenseqa_retention, commonsenseqa_verdict = compute_retention(commonsenseqa_base_acc, commonsenseqa_instruct_acc)
print(f"\n{'─'*40}")
print(f"CommonsenseQA  instruct_acc : {commonsenseqa_instruct_acc:.4f}")
print(f"CommonsenseQA  base_acc     : {commonsenseqa_base_acc:.4f}")
print(f"CommonsenseQA  retention    : {commonsenseqa_retention:.4f}")
print(f"CommonsenseQA  verdict      : {commonsenseqa_verdict}")

In [ ]:
import json, datetime
from pathlib import Path

results = {
    "experiment": "H2-retention-MCQ",
    "date": datetime.datetime.utcnow().isoformat(),
    "hardware": HW_META,
    "protocol": {
        "n_eval": N_EVAL,
        "max_new_tokens": MAX_NEW_TOKENS_MCQ,
        "decoding": "greedy",
        "instruct_model": INSTRUCT_MODEL_ID,
        "base_model": BASE_MODEL_ID,
        "tokenizer": TOKENIZER_ID,
        "pass_threshold": 0.70,
        "fail_threshold": 0.30,
    },
    "tasks": {
        "arc-easy": {
            "phase1_best_reward": 1.0000,
            "instruct_acc": arc_easy_instruct_acc,
            "base_acc": arc_easy_base_acc,
            "retention": arc_easy_retention,
            "verdict": arc_easy_verdict,
        },
        "arc-challenge": {
            "phase1_best_reward": None,
            "instruct_acc": arc_challenge_instruct_acc,
            "base_acc": arc_challenge_base_acc,
            "retention": arc_challenge_retention,
            "verdict": arc_challenge_verdict,
        },
        "openbookqa": {
            "phase1_best_reward": 0.9250,
            "instruct_acc": openbookqa_instruct_acc,
            "base_acc": openbookqa_base_acc,
            "retention": openbookqa_retention,
            "verdict": openbookqa_verdict,
        },
        "commonsenseqa": {
            "phase1_best_reward": None,
            "instruct_acc": commonsenseqa_instruct_acc,
            "base_acc": commonsenseqa_base_acc,
            "retention": commonsenseqa_retention,
            "verdict": commonsenseqa_verdict,
        },
    },
}

output_path = Path("/kaggle/working/h2_retention_results_mcq.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*60)
print("H2 RETENTION RESULTS — MCQ BATCH")
print("="*60)
for task, v in results["tasks"].items():
    ret = v['retention']
    ret_str = f"{ret:.4f}" if ret is not None else "N/A"
    print(f"  {task:<20} instruct={v['instruct_acc']:.4f}  base={v['base_acc']:.4f}  "
          f"retention={ret_str}  [{v['verdict']}]")
print("="*60)
print(f"\nResults saved to: {output_path}")
print(json.dumps(results, indent=2))